# EA3 — Proyecto final: procesamiento distribuido y cierre del caso

**Curso:** Procesamiento distribuido de datos  
**Caso:** análisis de ventas sobre arquitectura Lakehouse / Delta  
**Integrante:** Esteban Duarte Salgado

**Repositorio:** `https://github.com/ED-SALGADO-IUD/final-data-brick`  
**Video de sustentación:** `CAMBIAR_AQUI_LINK_VIDEO`

> Este notebook continúa el mismo caso trabajado en EA1 y EA2. Utiliza las tablas existentes
> `ea2_bronze.ventas_raw` y `ea2_silver.ventas_limpias`, construye/actualiza la capa oro,
> documenta el recorrido del dato y realiza una optimización medida con Spark.


## 1. Contexto, objetivo y configuración

La pregunta de negocio elegida para la capa oro es:

**¿Cómo se distribuyen los ingresos, las unidades vendidas y el número de transacciones completadas por año, mes, país y categoría de producto?**

La capa oro debe poder ser interpretada directamente por un usuario de negocio: utiliza nombres descriptivos,
métricas calculadas y se presenta ordenada por ingresos de mayor a menor.


In [0]:
from statistics import median
from time import perf_counter

catalog = spark.sql("SELECT current_catalog() AS catalog").first()["catalog"]

def identificar(nombre):
    return f"`{str(nombre).replace('`', '``')}`"

catalog_sql = identificar(catalog)

tabla_bronze = f"{catalog_sql}.ea2_bronze.ventas_raw"
tabla_silver = f"{catalog_sql}.ea2_silver.ventas_limpias"
tabla_gold = f"{catalog_sql}.ea2_gold.resumen_ventas"

print("Catálogo actual:", catalog)
print("Bronze:", tabla_bronze)
print("Silver:", tabla_silver)
print("Gold:", tabla_gold)

# Validación temprana: el notebook se detiene con un mensaje claro si falta alguna tabla base.
for nombre_tabla in [tabla_bronze, tabla_silver]:
    if not spark.catalog.tableExists(nombre_tabla):
        raise RuntimeError(
            f"No existe {nombre_tabla}. Ejecuta primero el pipeline de EA2 que crea Bronze y Silver."
        )

print("Validación de tablas base: OK")


Catálogo actual: workspace
Bronze: `workspace`.ea2_bronze.ventas_raw
Silver: `workspace`.ea2_silver.ventas_limpias
Gold: `workspace`.ea2_gold.resumen_ventas
Validación de tablas base: OK


## 2. Cierre del pipeline: construcción de la capa oro

Se conserva únicamente el estado `COMPLETADA`, porque la pregunta de negocio busca medir ventas efectivamente realizadas.
La agregación se hace por tiempo, país y categoría para producir una tabla compacta de consumo analítico.


In [0]:
spark.sql(f'''
CREATE OR REPLACE TABLE {tabla_gold}
USING DELTA
AS
SELECT
    YEAR(sale_date) AS anio,
    MONTH(sale_date) AS mes,
    country AS pais,
    product_category AS categoria_producto,
    COUNT(DISTINCT transaction_id) AS transacciones_completadas,
    SUM(quantity) AS unidades_vendidas,
    CAST(SUM(total_amount) AS DECIMAL(18,2)) AS ingresos_totales,
    CAST(AVG(total_amount) AS DECIMAL(18,2)) AS ticket_promedio
FROM {tabla_silver}
WHERE status = 'COMPLETADA'
GROUP BY
    YEAR(sale_date),
    MONTH(sale_date),
    country,
    product_category
''')

print("Capa oro creada/actualizada correctamente.")
display(
    spark.sql(f'''
        SELECT *
        FROM {tabla_gold}
        ORDER BY ingresos_totales DESC, anio, mes, pais, categoria_producto
    ''')
)


Capa oro creada/actualizada correctamente.


anio,mes,pais,categoria_producto,transacciones_completadas,unidades_vendidas,ingresos_totales,ticket_promedio
2026,8,Colombia,Tecnologia,4,9,14990.00,3747.50
2026,8,Peru,Tecnologia,2,4,3820.00,1910.00
2026,8,Colombia,Accesorios,5,23,2850.00,570.00
2026,8,Ecuador,Tecnologia,1,2,1700.00,1700.00
2026,8,Peru,Accesorios,1,8,480.00,480.00


## 3. Recorrido del dato: Bronze → Silver → Gold

La diferencia entre Bronze y Silver corresponde al proceso de calidad aplicado en EA2:
deduplicación por `transaction_id` conservando la versión más reciente y exclusión de registros con
`quantity <= 0` o `unit_price < 0`.

Para pasar de Silver a Gold primero se consideran únicamente las ventas `COMPLETADA`.
Después, esas filas se consolidan en grupos de año, mes, país y categoría; por eso el número de filas de Gold
es menor que el número de transacciones completadas.


In [0]:
conteo_bronze = spark.sql(f"SELECT COUNT(*) AS n FROM {tabla_bronze}").first()["n"]
conteo_silver = spark.sql(f"SELECT COUNT(*) AS n FROM {tabla_silver}").first()["n"]
conteo_completadas = spark.sql(
    f"SELECT COUNT(*) AS n FROM {tabla_silver} WHERE status = 'COMPLETADA'"
).first()["n"]
conteo_gold = spark.sql(f"SELECT COUNT(*) AS n FROM {tabla_gold}").first()["n"]

descartados_calidad = conteo_bronze - conteo_silver
no_completadas = conteo_silver - conteo_completadas
reduccion_agregacion = conteo_completadas - conteo_gold

recorrido = spark.createDataFrame([
    ("Bronze", int(conteo_bronze), "Datos crudos ingeridos"),
    ("Silver", int(conteo_silver), "Datos deduplicados y con validaciones de cantidad/precio"),
    ("Silver COMPLETADA", int(conteo_completadas), "Ventas efectivamente completadas"),
    ("Gold", int(conteo_gold), "Agregación por año, mes, país y categoría"),
], ["capa", "registros", "interpretacion"])

display(recorrido)

print(f"Registros descartados/absorbidos por reglas de calidad Bronze→Silver: {descartados_calidad}")
print(f"Registros Silver no completados excluidos de la capa Gold: {no_completadas}")
print(f"Reducción adicional por agregación de transacciones completadas: {reduccion_agregacion}")
print(f"Filas finales de la capa Gold: {conteo_gold}")


capa,registros,interpretacion
Bronze,16,Datos crudos ingeridos
Silver,15,Datos deduplicados y con validaciones de cantidad/precio
Silver COMPLETADA,13,Ventas efectivamente completadas
Gold,5,"Agregación por año, mes, país y categoría"


Registros descartados/absorbidos por reglas de calidad Bronze→Silver: 1
Registros Silver no completados excluidos de la capa Gold: 2
Reducción adicional por agregación de transacciones completadas: 8
Filas finales de la capa Gold: 5


## 4. Operación distribuida elegida y plan inicial

Para medir una operación con costo distribuido se cruza la tabla detallada `ventas_limpias`
con la tabla agregada `resumen_ventas`.

El cruce enriquece cada transacción completada con las métricas del segmento al que pertenece
(año, mes, país y categoría). Esto obliga a Spark a resolver un `join` por varias claves.

### Técnica evaluada

Se probará **Broadcast Hash Join** sobre la capa Gold, porque Gold es una tabla agregada y normalmente mucho
más pequeña que Silver. En un escenario distribuido, enviar esa tabla pequeña a los ejecutores puede evitar
el `shuffle` y el ordenamiento de ambos lados del `join`.

Para hacer visible la comparación sin modificar configuraciones globales de Spark, el plan inicial utiliza
un `MERGE hint`, mientras que el plan optimizado aplica explícitamente `broadcast()` sobre la tabla Gold.
Esta estrategia es compatible con el entorno Serverless utilizado en Databricks.


In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(tabla_silver)
gold_df = spark.table(tabla_gold)

# Se usan únicamente ventas completadas, igual que en la construcción de Gold.
silver_join = (
    silver_df
    .filter(F.col("status") == "COMPLETADA")
    .withColumn("anio", F.year("sale_date"))
    .withColumn("mes", F.month("sale_date"))
)

# Misma lógica y mismas columnas que la consulta optimizada.
# La única diferencia será la estrategia física del join.
consulta_inicial = (
    silver_join.alias("s")
    .join(
        gold_df.hint("merge").alias("g"),
        on=[
            F.col("s.anio") == F.col("g.anio"),
            F.col("s.mes") == F.col("g.mes"),
            F.col("s.country") == F.col("g.pais"),
            F.col("s.product_category") == F.col("g.categoria_producto")
        ],
        how="inner"
    )
    .select(
        F.col("s.transaction_id"),
        F.col("s.sale_date"),
        F.col("s.customer_name"),
        F.col("s.product_name"),
        F.col("s.total_amount"),
        F.col("g.transacciones_completadas"),
        F.col("g.unidades_vendidas"),
        F.col("g.ingresos_totales"),
        F.col("g.ticket_promedio")
    )
)

print("=== PLAN DE EJECUCIÓN INICIAL ===")
consulta_inicial.explain("formatted")


=== PLAN DE EJECUCIÓN INICIAL ===
== Physical Plan ==
AdaptiveSparkPlan (18)
+- == Initial Plan ==
   Project (17)
   +- SortMergeJoin Inner (16)
      :- ColumnarToRow (8)
      :  +- PhotonResultStage (7)
      :     +- PhotonSort (6)
      :        +- PhotonShuffleExchangeSource (5)
      :           +- PhotonShuffleMapStage (4)
      :              +- PhotonShuffleExchangeSink (3)
      :                 +- PhotonProject (2)
      :                    +- PhotonScan parquet workspace.ea2_silver.ventas_limpias (1)
      +- ColumnarToRow (15)
         +- PhotonResultStage (14)
            +- PhotonSort (13)
               +- PhotonShuffleExchangeSource (12)
                  +- PhotonShuffleMapStage (11)
                     +- PhotonShuffleExchangeSink (10)
                        +- PhotonScan parquet workspace.ea2_gold.resumen_ventas (9)


(1) PhotonScan parquet workspace.ea2_silver.ventas_limpias
Output [8]: [transaction_id#22622, sale_date#22623, customer_name#22625, country#2262

### Lectura esperada del plan inicial

Con el `MERGE hint`, el plan inicial debe favorecer un **SortMergeJoin**. En ese tipo de estrategia suelen aparecer
operaciones **Exchange** para redistribuir los datos por las claves de cruce y operaciones de ordenamiento previas al `join`.

El costo relevante no es únicamente el `join`: también importa el movimiento de datos por red y el ordenamiento necesario
para resolverlo. El plan visible en la salida del notebook es la evidencia que se utilizará para comparar con la versión optimizada.


In [0]:
from time import perf_counter
from statistics import median

def medir_consulta(df, etiqueta):
    print(f"\n--- {etiqueta}: calentamiento (NO se reporta como medición) ---")

    t0 = perf_counter()
    filas_warmup = df.count()
    warmup = perf_counter() - t0

    print(f"Calentamiento: {warmup:.6f} s | filas={filas_warmup}")

    tiempos = []

    for i in range(1, 4):
        t0 = perf_counter()
        filas = df.count()
        elapsed = perf_counter() - t0

        tiempos.append(elapsed)

        print(f"Medición {i}: {elapsed:.6f} s | filas={filas}")

    med = median(tiempos)

    print(f"Mediana reportada ({etiqueta}): {med:.6f} s")

    return tiempos, med, filas_warmup


tiempos_antes, mediana_antes, filas_antes = medir_consulta(
    consulta_inicial,
    "ANTES de optimizar"
)


--- ANTES de optimizar: calentamiento (NO se reporta como medición) ---
Calentamiento: 1.094083 s | filas=13
Medición 1: 0.832438 s | filas=13
Medición 2: 0.924771 s | filas=13
Medición 3: 0.831716 s | filas=13
Mediana reportada (ANTES de optimizar): 0.832438 s


## 5. Optimización: Broadcast Hash Join

La única técnica de optimización aplicada es **broadcast de la tabla Gold**.

La lógica de negocio de la consulta no cambia. Solo cambia la estrategia física:
la tabla pequeña se replica en los ejecutores para que cada partición de Silver pueda resolver el cruce localmente,
reduciendo la necesidad de redistribuir y ordenar ambos conjuntos de datos.


In [0]:
from pyspark.sql.functions import broadcast

consulta_optimizada = (
    silver_join.alias("s")
    .join(
        broadcast(gold_df).alias("g"),
        on=[
            F.col("s.anio") == F.col("g.anio"),
            F.col("s.mes") == F.col("g.mes"),
            F.col("s.country") == F.col("g.pais"),
            F.col("s.product_category") == F.col("g.categoria_producto")
        ],
        how="inner"
    )
    .select(
        F.col("s.transaction_id"),
        F.col("s.sale_date"),
        F.col("s.customer_name"),
        F.col("s.product_name"),
        F.col("s.total_amount"),
        F.col("g.transacciones_completadas"),
        F.col("g.unidades_vendidas"),
        F.col("g.ingresos_totales"),
        F.col("g.ticket_promedio")
    )
)

print("=== PLAN DE EJECUCIÓN DESPUÉS DE LA OPTIMIZACIÓN ===")
consulta_optimizada.explain("formatted")


=== PLAN DE EJECUCIÓN DESPUÉS DE LA OPTIMIZACIÓN ===
== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonProject (2)
            :  +- PhotonScan parquet workspace.ea2_silver.ventas_limpias (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonScan parquet workspace.ea2_gold.resumen_ventas (3)


(1) PhotonScan parquet workspace.ea2_silver.ventas_limpias
Output [8]: [transaction_id#22762, sale_date#22763, customer_name#22765, country#22766, product_category#22767, product_name#22768, status#22771, total_amount#22773]
DictionaryFilters: [(status#22771 = COMPLETADA)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-sch7d/uc/ef7b8f61-446d-46ed-8a17-0d8e60ec8e63/277a78fb-a630-4f93-9218-9c4dda5e3ef0/__unity

In [0]:
tiempos_despues, mediana_despues, filas_despues = medir_consulta(
    consulta_optimizada,
    "DESPUÉS de optimizar"
)

if mediana_antes > 0:
    mejora_pct = ((mediana_antes - mediana_despues) / mediana_antes) * 100
else:
    mejora_pct = 0.0

factor = (mediana_antes / mediana_despues) if mediana_despues > 0 else float("inf")

comparacion = spark.createDataFrame([
    ("Antes", float(tiempos_antes[0]), float(tiempos_antes[1]), float(tiempos_antes[2]), float(mediana_antes)),
    ("Después", float(tiempos_despues[0]), float(tiempos_despues[1]), float(tiempos_despues[2]), float(mediana_despues)),
], ["escenario", "medicion_1_s", "medicion_2_s", "medicion_3_s", "mediana_s"])

display(comparacion)

print(f"Filas producidas antes: {filas_antes}")
print(f"Filas producidas después: {filas_despues}")
print(f"Mediana antes: {mediana_antes:.6f} s")
print(f"Mediana después: {mediana_despues:.6f} s")
print(f"Variación porcentual: {mejora_pct:.2f}%")
print(f"Factor antes/después: {factor:.3f}x")



--- DESPUÉS de optimizar: calentamiento (NO se reporta como medición) ---
Calentamiento: 0.998436 s | filas=13
Medición 1: 0.861553 s | filas=13
Medición 2: 0.766834 s | filas=13
Medición 3: 0.873468 s | filas=13
Mediana reportada (DESPUÉS de optimizar): 0.861553 s


escenario,medicion_1_s,medicion_2_s,medicion_3_s,mediana_s
Antes,0.8324383199999943,0.9247706659999722,0.8317157930000008,0.8324383199999943
Después,0.861553291000007,0.7668339929999775,0.8734680719999801,0.861553291000007


Filas producidas antes: 13
Filas producidas después: 13
Mediana antes: 0.832438 s
Mediana después: 0.861553 s
Variación porcentual: -3.50%
Factor antes/después: 0.966x


In [0]:
import io
from contextlib import redirect_stdout


def obtener_plan(df):
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        df.explain("formatted")
    return buffer.getvalue()


plan_antes = obtener_plan(consulta_inicial)
plan_despues = obtener_plan(consulta_optimizada)


def contiene(plan, termino):
    return termino.lower() in plan.lower()


print("=== COMPARACIÓN AUTOMÁTICA DE LOS PLANES ===")
print("ANTES  - SortMergeJoin:", contiene(plan_antes, "SortMergeJoin"))
print("ANTES  - BroadcastHashJoin:", contiene(plan_antes, "BroadcastHashJoin"))
print("ANTES  - Exchange:", contiene(plan_antes, "Exchange"))
print("DESPUÉS - SortMergeJoin:", contiene(plan_despues, "SortMergeJoin"))
print("DESPUÉS - BroadcastHashJoin:", contiene(plan_despues, "BroadcastHashJoin"))
print("DESPUÉS - Exchange:", contiene(plan_despues, "Exchange"))


if mediana_antes > 0:
    mejora_pct = ((mediana_antes - mediana_despues) / mediana_antes) * 100
else:
    mejora_pct = 0.0


if mediana_despues < mediana_antes:
    conclusion = (
        f"La optimización redujo la mediana de {mediana_antes:.6f} s a "
        f"{mediana_despues:.6f} s, una mejora de {mejora_pct:.2f}%. "
        "El plan optimizado utiliza BroadcastHashJoin para distribuir la tabla Gold, "
        "que es considerablemente más pequeña. Esto reduce la necesidad de redistribuir "
        "y ordenar ambos lados del join respecto al plan inicial."
    )
elif mediana_despues > mediana_antes:
    conclusion = (
        f"La mediana pasó de {mediana_antes:.6f} s a {mediana_despues:.6f} s; "
        f"la variación fue {mejora_pct:.2f}%. En este volumen la técnica de broadcast no produjo "
        "una mejora temporal. Esto puede ocurrir cuando el conjunto es pequeño, el costo de broadcast "
        "domina el tiempo total o el ruido de ejecución es comparable al costo de la consulta. "
        "El resultado se reporta sin modificar las mediciones."
    )
else:
    conclusion = (
        f"Las medianas fueron iguales ({mediana_antes:.6f} s). "
        "La técnica modificó el mecanismo físico del join, pero en este volumen la diferencia temporal "
        "no fue distinguible del ruido de ejecución."
    )


print("\n=== INTERPRETACIÓN DE RESULTADOS ===")
print(conclusion)

print("\n=== RESUMEN DE MEDICIONES ===")
print(f"Mediana ANTES:   {mediana_antes:.6f} s")
print(f"Mediana DESPUÉS: {mediana_despues:.6f} s")
print(f"Variación porcentual: {mejora_pct:.2f}%")


=== COMPARACIÓN AUTOMÁTICA DE LOS PLANES ===
ANTES  - SortMergeJoin: True
ANTES  - BroadcastHashJoin: False
ANTES  - Exchange: True
DESPUÉS - SortMergeJoin: False
DESPUÉS - BroadcastHashJoin: True
DESPUÉS - Exchange: True

=== INTERPRETACIÓN DE RESULTADOS ===
La mediana pasó de 0.832438 s a 0.861553 s; la variación fue -3.50%. En este volumen la técnica de broadcast no produjo una mejora temporal. Esto puede ocurrir cuando el conjunto es pequeño, el costo de broadcast domina el tiempo total o el ruido de ejecución es comparable al costo de la consulta. El resultado se reporta sin modificar las mediciones.

=== RESUMEN DE MEDICIONES ===
Mediana ANTES:   0.832438 s
Mediana DESPUÉS: 0.861553 s
Variación porcentual: -3.50%


### ¿Por qué puede mejorar?

En el plan inicial, un `SortMergeJoin` suele necesitar particionar y ordenar ambos lados por las claves de cruce.
Eso genera `Exchange` y movimiento de datos entre ejecutores. Con `BroadcastHashJoin`, la tabla Gold —que es agregada
y pequeña respecto a Silver— se envía a los ejecutores y se construye una estructura hash local. De esta forma,
las particiones de Silver pueden resolver el cruce sin redistribuir también el lado grande.

### ¿Cuándo sería contraproducente?

El broadcast deja de ser apropiado cuando la tabla que se intenta distribuir ya no es pequeña.
Si Gold creciera hasta ocupar cientos de megabytes o gigabytes, replicarla en cada ejecutor aumentaría el uso de memoria,
el tráfico de red y el riesgo de `OutOfMemory`. En ese escenario podría ser mejor permitir un `SortMergeJoin`,
reparticionar por las claves correctas o aplicar otras estrategias orientadas a tablas grandes.

También puede no aportar una mejora medible cuando el volumen total es pequeño: el costo fijo de preparar el broadcast
puede ser similar o superior al ahorro conseguido.


## 6. Cierre del caso y pregunta orientadora

**Pregunta:**  
Una empresa recibe datos de su portal web, Facebook, Instagram, TikTok y WhatsApp Business.
¿Bajo qué paradigma, con qué herramientas y con qué arquitectura debería analizarlos?

Para este caso conviene un paradigma de **procesamiento distribuido sobre un Lakehouse**. Las fuentes son heterogéneas:
pueden entregar eventos web, datos tabulares, JSON, métricas de campañas y mensajes o metadatos semiestructurados.
Centralizar primero los datos en una capa Bronze permite conservar la evidencia original; transformarlos a Silver
permite estandarizar tipos, nombres, calidad y reglas de negocio; y una capa Gold produce métricas listas para consumo.
Este patrón es el mismo que se aplicó en el proyecto con las ventas: datos crudos en Bronze, depuración en Silver
y agregados entendibles para negocio en Gold.

Como herramientas, **Apache Spark** permite procesar el volumen de manera distribuida; **Delta Lake** aporta tablas
transaccionales y una base consistente para las capas; y **Databricks** integra ejecución, catálogo, gobierno,
orquestación y consumo sobre el mismo entorno. La arquitectura recomendada es:
**fuentes digitales → ingesta → Bronze/Delta → transformación Spark → Silver/Delta → agregaciones Gold/Delta → BI/analítica**.
La optimización realizada en este notebook muestra además que no basta con ejecutar consultas: la estrategia física
del procesamiento —por ejemplo, evitar shuffle innecesario mediante broadcast cuando una dimensión es pequeña—
también debe ajustarse al tamaño y forma real de los datos.

### Conclusiones del proyecto

El pipeline por capas funcionó porque separó responsabilidades: Bronze preservó la entrada, Silver concentró la calidad
y Gold respondió una pregunta concreta de negocio. Los conteos entre capas permiten explicar qué ocurrió con los registros
en lugar de tratar la reducción de filas como una pérdida inexplicable.

La medición de rendimiento también mostró que una optimización debe evaluarse con evidencia: calentamiento descartado,
tres ejecuciones y mediana. Si el broadcast mejora, el cambio se explica por la eliminación/reducción de shuffle y
ordenamiento; si no mejora, el resultado sigue siendo válido porque el volumen o el costo fijo pueden hacer que la
diferencia quede dentro del ruido. Si el proyecto empezara de nuevo, se definirían desde el comienzo métricas de calidad
por capa y pruebas de rendimiento repetibles para que el comportamiento del pipeline sea verificable desde sus primeras etapas.


## 7. Contribución, integridad y uso de IA

**Contribución declarada — Esteban Duarte Salgado**

- Continuidad del pipeline del caso de ventas.
- Construcción y validación de la capa Gold.
- Conteo y explicación del recorrido Bronze → Silver → Gold.
- Ejecución del benchmark antes y después de la optimización.
- Revisión de los planes de ejecución y sustentación del uso de Broadcast Hash Join.
- Cierre de la pregunta orientadora y conclusiones.

**Uso de asistentes de IA:** se utilizó ChatGPT como apoyo para estructurar y revisar el notebook,
organizar el procedimiento de medición y mejorar la explicación técnica. Las consultas, mediciones, planes y resultados
que aparecen en la entrega deben corresponder a la ejecución real en Databricks y ser explicables por el integrante
durante la sustentación.

### Verificación final antes de exportar

- [ ] El notebook ejecuta de arriba abajo sin errores.
- [ ] La tabla Gold aparece con datos.
- [ ] Los conteos Bronze, Silver y Gold son visibles.
- [ ] Hay un calentamiento descartado y tres mediciones visibles ANTES.
- [ ] Hay un calentamiento descartado y tres mediciones visibles DESPUÉS.
- [ ] Las dos medianas están visibles.
- [ ] El plan inicial está visible.
- [ ] El plan optimizado está visible.
- [ ] La interpretación automática de resultados está visible.
- [ ] El enlace de GitHub fue reemplazado en la portada.
- [ ] El enlace del video fue reemplazado en la portada y abre desde otra cuenta.
- [ ] El notebook `.ipynb` está en `/ea3` del repositorio.
- [ ] Después de ejecutar todo, se exportó a HTML desde Databricks.
